# 🗺️ OSM Tile Extractor for Intersection Detection

This notebook extracts OpenStreetMap tiles systematically for training machine learning models to detect road intersections.

## Features
- 📊 **Visual Progress Tracking** - Real-time charts and progress bars
- 🗺️ **Interactive Preview Maps** - See your extraction area before downloading
- ⚡ **Fast Concurrent Downloads** - Multiple servers with rate limiting
- 💾 **Resume Capability** - Pick up where you left off
- 📈 **Performance Analytics** - Speed, success rates, and statistics

## 1. Setup & Dependencies

In [1]:
# Install required packages (run this first)
!pip install aiohttp folium tqdm pandas matplotlib seaborn requests ipywidgets

print("✅ Dependencies installed!")

✅ Dependencies installed!


In [ ]:
# Import all required modules including visualization
import os
import time
import sqlite3
import asyncio
from pathlib import Path
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Core extraction system imports
from simple_grid_coverage import SimpleGridCoverage, GridSquare
from fast_osm_extractor import FastOSMExtractor

# Modular function imports
from grid_extraction import (
    extract_single_grid,
    extract_multiple_grids,
    extract_coastal_grids,
    extract_grid_tiles,
    test_single_extraction
)

from grid_monitoring import (
    show_grid_progress,
    show_next_grids,
    show_state_statistics,
    show_recent_activity,
    quick_status,
    show_grid_details,
    show_extraction_summary,
    get_grid_recommendations
)

# NEW: Visualization imports
from grid_visualization import (
    GridVisualization,
    show_grid_map,
    save_grid_map,
    show_focused_map,
    show_progress_heatmap,
    show_california_map,
    show_texas_map,
    show_florida_map,
    show_northeast_map
)

from interactive_grid_map import (
    InteractiveGridMap,
    LiveProgressTracker,
    show_interactive_map,
    start_live_tracking
)

from osm_extraction_main import OSMExtractionManager

print("✅ All modules imported successfully!")
print("📦 Available systems:")
print("   • SimpleGridCoverage - Grid management")
print("   • FastOSMExtractor - Tile downloading")
print("   • grid_extraction - Extraction functions")
print("   • grid_monitoring - Progress monitoring")
print("   • grid_visualization - Static maps with color-coded grids")
print("   • interactive_grid_map - Interactive maps with real-time updates")
print("   • OSMExtractionManager - Complete system")

print("\n🗺️ NEW VISUALIZATION FEATURES:")
print("   • Color-coded grids: Red=Pending, Yellow=In Progress, Green=Completed")
print("   • Interactive maps with refresh buttons and auto-update")
print("   • Regional focus (California, Texas, Florida, Northeast)")
print("   • Progress heatmaps and live tracking")
print("   • Real-time map updates during extraction")

## 2. Configuration & Settings

In [ ]:
# Initialize the complete extraction system using the modular approach

# Configuration
ZOOM_LEVEL = 15
GRID_SIZE_KM = 25.0
OUTPUT_FOLDER = "osm_tiles"

print("🚀 Initializing OSM Extraction System...")
print(f"   Zoom Level: {ZOOM_LEVEL}")
print(f"   Grid Size: {GRID_SIZE_KM}km x {GRID_SIZE_KM}km")
print(f"   Output Folder: {OUTPUT_FOLDER}")

# Initialize the complete manager (easiest approach)
manager = OSMExtractionManager(
    zoom_level=ZOOM_LEVEL,
    grid_size_km=GRID_SIZE_KM,
    output_folder=OUTPUT_FOLDER
)

# Or if you prefer direct access to components:
coverage_system = manager.coverage_system
extractor = manager.extractor

print("✅ System initialization complete!")
print("\n📊 Quick Status Check:")
manager.show_status()

print("\n🎯 Available Functions:")
print("   EXTRACTION:")
print("   • await manager.extract_single() - Extract next grid")
print("   • await manager.extract_multiple(n) - Extract n grids")
print("   • await manager.extract_coastal(n) - Extract coastal grids")
print("   • await manager.test_extraction() - Test workflow")
print()
print("   MONITORING:")
print("   • manager.show_progress() - Full report")
print("   • manager.show_status() - Quick status")
print("   • manager.show_next(n) - Next n grids")
print("   • manager.show_states() - Progress by state")
print("   • manager.show_recent() - Recent activity")

## 3. Core Classes & Functions

In [ ]:
@dataclass
class TileInfo:
    """Information about a map tile"""
    x: int
    y: int
    zoom: int
    lat_min: float
    lat_max: float
    lon_min: float
    lon_max: float
    filename: str
    downloaded: bool = False
    timestamp: Optional[str] = None
    server_used: Optional[str] = None

@dataclass
class TileServer:
    """Configuration for a tile server"""
    name: str
    url_template: str
    subdomains: List[str]
    max_requests_per_second: float
    last_request_time: float = 0

@dataclass
class DownloadStats:
    """Statistics for download session"""
    total_tiles: int = 0
    downloaded: int = 0
    failed: int = 0
    skipped: int = 0
    start_time: float = 0
    end_time: float = 0
    speeds: List[float] = None
    
    def __post_init__(self):
        if self.speeds is None:
            self.speeds = []
    
    @property
    def success_rate(self) -> float:
        if self.total_tiles == 0:
            return 0.0
        return (self.downloaded / self.total_tiles) * 100
    
    @property
    def average_speed(self) -> float:
        return np.mean(self.speeds) if self.speeds else 0.0
    
    @property
    def total_time(self) -> float:
        return self.end_time - self.start_time if self.end_time > self.start_time else 0

print("📋 Core classes defined successfully!")

In [ ]:
class OSMTileExtractor:
    """Jupyter-optimized OSM tile extractor with visual progress tracking"""
    
    def __init__(self, output_dir: str = "osm_training_data", max_workers: int = 8):
        self.output_dir = Path(output_dir)
        self.max_workers = max_workers
        self.stats = DownloadStats()
        
        # Setup tile servers
        self.tile_servers = self._setup_tile_servers()
        self.server_rotation_index = 0
        
        # Progress tracking
        self.progress_bar = None
        self.live_stats = None
        
        self.setup_directories()
        self.init_database()
        
        print(f"🚀 OSM Extractor initialized:")
        print(f"   📁 Output directory: {self.output_dir}")
        print(f"   👥 Max workers: {self.max_workers}")
        print(f"   🌐 Tile servers: {len(self.tile_servers)}")
    
    def _setup_tile_servers(self) -> List[TileServer]:
        """Setup multiple tile servers for load balancing"""
        return [
            # OpenStreetMap servers
            TileServer("osm_main", "https://tile.openstreetmap.org/{z}/{x}/{y}.png", [""], 1.0),
            TileServer("osm_a", "https://a.tile.openstreetmap.org/{z}/{x}/{y}.png", [""], 1.0),
            TileServer("osm_b", "https://b.tile.openstreetmap.org/{z}/{x}/{y}.png", [""], 1.0),
            TileServer("osm_c", "https://c.tile.openstreetmap.org/{z}/{x}/{y}.png", [""], 1.0),
            
            # CartoDB (faster)
            TileServer("cartodb_light", "https://cartodb-basemaps-{s}.global.ssl.fastly.net/light_all/{z}/{x}/{y}.png", 
                      ["a", "b", "c", "d"], 3.0),
            
            # Stamen
            TileServer("stamen_toner", "https://stamen-tiles-{s}.a.ssl.fastly.net/toner/{z}/{x}/{y}.png", 
                      ["a", "b", "c", "d"], 2.0),
        ]
    
    def setup_directories(self):
        """Create organized directory structure"""
        directories = ["tiles", "metadata", "progress", "annotations", "processed"]
        for dir_name in directories:
            (self.output_dir / dir_name).mkdir(parents=True, exist_ok=True)
    
    def init_database(self):
        """Initialize SQLite database for progress tracking"""
        db_path = self.output_dir / "progress.db"
        with sqlite3.connect(db_path) as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS tiles (
                    id INTEGER PRIMARY KEY,
                    x INTEGER, y INTEGER, zoom INTEGER,
                    lat_min REAL, lat_max REAL, lon_min REAL, lon_max REAL,
                    filename TEXT, downloaded BOOLEAN DEFAULT FALSE,
                    timestamp TEXT, server_used TEXT,
                    UNIQUE(x, y, zoom)
                )
            """)
            conn.commit()
    
    def deg2num(self, lat_deg: float, lon_deg: float, zoom: int) -> Tuple[int, int]:
        """Convert lat/lon coordinates to tile numbers"""
        lat_rad = math.radians(lat_deg)
        n = 2.0 ** zoom
        x = int((lon_deg + 180.0) / 360.0 * n)
        y = int((1.0 - math.asinh(math.tan(lat_rad)) / math.pi) / 2.0 * n)
        return x, y
    
    def num2deg(self, x: int, y: int, zoom: int) -> Tuple[float, float, float, float]:
        """Convert tile numbers to lat/lon bounding box"""
        n = 2.0 ** zoom
        lon_min = x / n * 360.0 - 180.0
        lat_max = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * y / n))))
        lon_max = (x + 1) / n * 360.0 - 180.0
        lat_min = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * (y + 1) / n))))
        return lat_min, lat_max, lon_min, lon_max
    
    def generate_tile_grid(self, bbox: Tuple[float, float, float, float], zoom: int) -> List[TileInfo]:
        """Generate systematic grid of tiles for a bounding box"""
        lat_min, lat_max, lon_min, lon_max = bbox
        
        x_min, y_max = self.deg2num(lat_min, lon_min, zoom)
        x_max, y_min = self.deg2num(lat_max, lon_max, zoom)
        
        tiles = []
        for x in range(x_min, x_max + 1):
            for y in range(y_min, y_max + 1):
                tile_lat_min, tile_lat_max, tile_lon_min, tile_lon_max = self.num2deg(x, y, zoom)
                filename = f"tile_{zoom}_{x}_{y}.png"
                
                tile = TileInfo(
                    x=x, y=y, zoom=zoom,
                    lat_min=tile_lat_min, lat_max=tile_lat_max,
                    lon_min=tile_lon_min, lon_max=tile_lon_max,
                    filename=filename
                )
                tiles.append(tile)
        
        return tiles
    
    def get_next_server(self) -> TileServer:
        """Get next available server with load balancing"""
        current_time = time.time()
        
        for _ in range(len(self.tile_servers)):
            server = self.tile_servers[self.server_rotation_index]
            self.server_rotation_index = (self.server_rotation_index + 1) % len(self.tile_servers)
            
            min_interval = 1.0 / server.max_requests_per_second
            if current_time - server.last_request_time >= min_interval:
                server.last_request_time = current_time
                return server
        
        # If all servers are rate limited, use the least recently used
        return min(self.tile_servers, key=lambda s: s.last_request_time)
    
    def build_tile_url(self, server: TileServer, x: int, y: int, z: int) -> str:
        """Build tile URL with subdomain rotation"""
        subdomain = random.choice(server.subdomains) if server.subdomains else ""
        
        if "{s}" in server.url_template:
            return server.url_template.format(s=subdomain, x=x, y=y, z=z)
        else:
            return server.url_template.format(x=x, y=y, z=z)

print("🔧 OSMTileExtractor class defined successfully!")

## 4. Visualization & Preview Functions

In [ ]:
def create_preview_map(bbox: Tuple[float, float, float, float], zoom: int = 10) -> folium.Map:
    """Create an interactive map showing the extraction area"""
    lat_min, lat_max, lon_min, lon_max = bbox
    
    # Center point
    center_lat = (lat_min + lat_max) / 2
    center_lon = (lon_min + lon_max) / 2
    
    # Create map
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=zoom,
        tiles='OpenStreetMap'
    )
    
    # Add bounding box rectangle
    folium.Rectangle(
        bounds=[[lat_min, lon_min], [lat_max, lon_max]],
        popup=f"Extraction Area<br>Lat: {lat_min:.4f} to {lat_max:.4f}<br>Lon: {lon_min:.4f} to {lon_max:.4f}",
        tooltip="Click for area details",
        color='red',
        weight=2,
        fillOpacity=0.1
    ).add_to(m)
    
    # Add corner markers
    corners = [
        ([lat_min, lon_min], "SW Corner"),
        ([lat_min, lon_max], "SE Corner"),
        ([lat_max, lon_min], "NW Corner"),
        ([lat_max, lon_max], "NE Corner"),
        ([center_lat, center_lon], "Center")
    ]
    
    for (lat, lon), label in corners:
        folium.Marker(
            [lat, lon],
            popup=f"{label}<br>{lat:.4f}, {lon:.4f}",
            tooltip=label
        ).add_to(m)
    
    return m

def calculate_area_stats(bbox: Tuple[float, float, float, float], zoom: int) -> Dict:
    """Calculate statistics for the extraction area"""
    lat_min, lat_max, lon_min, lon_max = bbox
    
    # Approximate area in km²
    lat_diff = lat_max - lat_min
    lon_diff = lon_max - lon_min
    
    # Rough conversion (varies with latitude)
    avg_lat = (lat_min + lat_max) / 2
    km_per_degree_lat = 111.0
    km_per_degree_lon = 111.0 * math.cos(math.radians(avg_lat))
    
    area_km2 = lat_diff * lon_diff * km_per_degree_lat * km_per_degree_lon
    
    # Calculate number of tiles
    n = 2.0 ** zoom
    x_min = int((lon_min + 180.0) / 360.0 * n)
    x_max = int((lon_max + 180.0) / 360.0 * n)
    y_min = int((1.0 - math.asinh(math.tan(math.radians(lat_max))) / math.pi) / 2.0 * n)
    y_max = int((1.0 - math.asinh(math.tan(math.radians(lat_min))) / math.pi) / 2.0 * n)
    
    total_tiles = (x_max - x_min + 1) * (y_max - y_min + 1)
    
    # Estimate download time and storage
    avg_tile_size_kb = 15  # Typical OSM tile size
    estimated_storage_mb = total_tiles * avg_tile_size_kb / 1024
    estimated_storage_gb = estimated_storage_mb / 1024
    
    # Time estimates for different speeds
    slow_speed = 1.0  # tiles per second
    fast_speed = 10.0  # tiles per second with optimization
    
    slow_time_hours = total_tiles / slow_speed / 3600
    fast_time_hours = total_tiles / fast_speed / 3600
    
    return {
        'bbox': bbox,
        'zoom': zoom,
        'area_km2': area_km2,
        'total_tiles': total_tiles,
        'estimated_storage_mb': estimated_storage_mb,
        'estimated_storage_gb': estimated_storage_gb,
        'slow_download_hours': slow_time_hours,
        'fast_download_hours': fast_time_hours,
        'tile_dimensions': f"{x_max-x_min+1} x {y_max-y_min+1}"
    }

def display_area_stats(stats: Dict):
    """Display area statistics in a nice format"""
    print("📊 EXTRACTION AREA STATISTICS")
    print("=" * 40)
    print(f"📍 Bounding Box: {stats['bbox']}")
    print(f"🔍 Zoom Level: {stats['zoom']}")
    print(f"🗺️  Area: {stats['area_km2']:.1f} km²")
    print(f"🔢 Total Tiles: {stats['total_tiles']:,}")
    print(f"📐 Tile Grid: {stats['tile_dimensions']}")
    print(f"💾 Storage (Est.): {stats['estimated_storage_mb']:.1f} MB ({stats['estimated_storage_gb']:.2f} GB)")
    print("\n⏱️  DOWNLOAD TIME ESTIMATES:")
    print(f"   🐌 Slow (1 tile/sec): {stats['slow_download_hours']:.1f} hours")
    print(f"   🚀 Fast (10 tiles/sec): {stats['fast_download_hours']:.1f} hours")
    
    # Visual representation
    if stats['total_tiles'] < 1000:
        size_category = "🟢 SMALL - Good for testing"
    elif stats['total_tiles'] < 10000:
        size_category = "🟡 MEDIUM - Good for development"
    elif stats['total_tiles'] < 100000:
        size_category = "🟠 LARGE - Production dataset"
    else:
        size_category = "🔴 VERY LARGE - Consider smaller areas first"
    
    print(f"\n📈 Size Category: {size_category}")

print("📊 Visualization functions defined successfully!")

## 5. Progress Tracking & Analytics

In [ ]:
def create_progress_dashboard(stats: DownloadStats):
    """Create a live progress dashboard"""
    
    # Create subplots
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('📊 OSM Tile Download Dashboard', fontsize=16, fontweight='bold')
    
    # 1. Progress pie chart
    if stats.total_tiles > 0:
        sizes = [stats.downloaded, stats.failed, stats.total_tiles - stats.downloaded - stats.failed]
        labels = ['Downloaded', 'Failed', 'Remaining']
        colors = ['#2ecc71', '#e74c3c', '#95a5a6']
        
        ax1.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
        ax1.set_title('Overall Progress')
    
    # 2. Success rate over time
    if len(stats.speeds) > 0:
        ax2.plot(stats.speeds, color='#3498db', linewidth=2)
        ax2.set_title('Download Speed (tiles/sec)')
        ax2.set_xlabel('Batch Number')
        ax2.set_ylabel('Speed')
        ax2.grid(True, alpha=0.3)
    
    # 3. Statistics bar chart
    metrics = ['Downloaded', 'Failed', 'Success Rate %']
    values = [stats.downloaded, stats.failed, stats.success_rate]
    colors_bar = ['#2ecc71', '#e74c3c', '#f39c12']
    
    bars = ax3.bar(metrics, values, color=colors_bar)
    ax3.set_title('Download Statistics')
    ax3.set_ylabel('Count / Percentage')
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                f'{value:.0f}', ha='center', va='bottom')
    
    # 4. Time remaining estimate
    if stats.average_speed > 0 and stats.total_tiles > stats.downloaded + stats.failed:
        remaining_tiles = stats.total_tiles - stats.downloaded - stats.failed
        eta_seconds = remaining_tiles / stats.average_speed
        eta_hours = eta_seconds / 3600
        
        ax4.text(0.5, 0.7, f'⏱️ ETA: {eta_hours:.1f} hours', 
                ha='center', va='center', fontsize=14, fontweight='bold')
        ax4.text(0.5, 0.5, f'🚀 Avg Speed: {stats.average_speed:.1f} tiles/sec', 
                ha='center', va='center', fontsize=12)
        ax4.text(0.5, 0.3, f'📦 Remaining: {remaining_tiles:,} tiles', 
                ha='center', va='center', fontsize=12)
    
    ax4.set_xlim(0, 1)
    ax4.set_ylim(0, 1)
    ax4.axis('off')
    ax4.set_title('Time Estimates')
    
    plt.tight_layout()
    plt.show()

def save_session_report(stats: DownloadStats, output_dir: Path, bbox: Tuple):
    """Save a detailed session report"""
    report = {
        'session_info': {
            'timestamp': datetime.now().isoformat(),
            'bbox': bbox,
            'total_tiles': stats.total_tiles,
            'downloaded': stats.downloaded,
            'failed': stats.failed,
            'success_rate': stats.success_rate,
            'total_time_seconds': stats.total_time,
            'average_speed': stats.average_speed
        },
        'performance_data': {
            'speeds': stats.speeds,
            'max_speed': max(stats.speeds) if stats.speeds else 0,
            'min_speed': min(stats.speeds) if stats.speeds else 0
        }
    }
    
    report_path = output_dir / "progress" / f"session_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)
    
    print(f"📄 Session report saved: {report_path}")
    return report_path

print("📈 Progress tracking functions defined successfully!")

## 6. Interactive Area Selection

In [ ]:
# Easy-to-use extraction functions
# These functions use the imported modules for clean, simple extraction

# Quick extraction functions
async def extract_next_grid():
    """Extract the next available grid"""
    print("🎯 Extracting next grid...")
    result = await manager.extract_single()
    if result:
        print(f"✅ Success: {result[0]:,}/{result[1]:,} tiles extracted")
    return result

async def extract_multiple_grids_demo(num_grids=3):
    """Extract multiple grids with progress tracking"""
    print(f"🚀 Extracting {num_grids} grids...")
    result = await manager.extract_multiple(num_grids)
    print(f"🎉 Batch complete: {result[0]:,}/{result[1]:,} total tiles")
    return result

async def extract_coastal_grids_demo(num_grids=2):
    """Extract coastal grids for intersection variety"""
    print(f"🌊 Extracting {num_grids} coastal grids...")
    result = await manager.extract_coastal(num_grids)
    print(f"🏖️ Coastal extraction complete: {result[0]:,}/{result[1]:,} tiles")
    return result

# Monitoring functions
def check_progress():
    """Quick progress check"""
    manager.show_status()

def full_report():
    """Show comprehensive progress report"""
    manager.show_progress()

def show_upcoming_grids(count=5):
    """Show next grids to be extracted"""
    manager.show_next(count)

def show_state_breakdown():
    """Show progress by state"""
    manager.show_states()

def show_recent_work():
    """Show recent extraction activity"""
    manager.show_recent()

# Grid management functions
def initialize_fresh_grid():
    """Initialize a fresh grid system (WARNING: clears existing progress!)"""
    print("⚠️  WARNING: This will reset all progress!")
    confirm = input("Type 'YES' to confirm: ")
    if confirm == 'YES':
        return manager.initialize_grid_system()
    else:
        print("❌ Cancelled")
        return None

def get_grid_recommendations_demo(priority="coastal"):
    """Get recommended grids for extraction"""
    recs = manager.get_recommendations(priority)
    print(f"🎯 RECOMMENDED GRIDS ({priority}):")
    for i, grid_id in enumerate(recs[:10], 1):
        print(f"  {i:2}. {grid_id}")
    return recs

print("✅ Extraction control functions loaded!")
print("\n🎮 QUICK COMMANDS:")
print("   • await extract_next_grid() - Extract one grid")
print("   • await extract_multiple_grids_demo(n) - Extract n grids")
print("   • await extract_coastal_grids_demo(n) - Extract coastal grids")
print("   • check_progress() - Quick status")
print("   • full_report() - Complete report")
print("   • show_upcoming_grids(n) - Show next n grids")
print("   • get_grid_recommendations_demo() - Get recommendations")

## 7. Control Panel

Use this interactive control panel to select your extraction area and settings:

In [ ]:
# 🗺️ VISUAL GRID CONTROL PANEL
# Interactive widgets with live map visualization

print("🎮 SETTING UP VISUAL CONTROL PANEL...")

# Create the main visual interface
def create_visual_interface():
    """Create the complete visual interface with maps and controls"""
    
    # Header
    header = widgets.HTML("""
    <h1 style='text-align: center; color: #2E86AB; margin-bottom: 20px;'>
        🗺️ OSM Grid Extraction Visual Control Center
    </h1>
    <p style='text-align: center; font-size: 16px; color: #666;'>
        Interactive map with color-coded grid progress tracking
    </p>
    """)
    
    # Quick action buttons
    map_buttons = widgets.HBox([
        widgets.Button(description='🗺️ Show Full Map', button_style='info', layout=widgets.Layout(width='140px')),
        widgets.Button(description='🔄 Refresh Map', button_style='success', layout=widgets.Layout(width='140px')),
        widgets.Button(description='📸 Save Map', button_style='warning', layout=widgets.Layout(width='140px')),
        widgets.Button(description='🌡️ Heatmap', button_style='danger', layout=widgets.Layout(width='140px'))
    ])
    
    # Regional focus buttons
    region_buttons = widgets.HBox([
        widgets.Button(description='🌴 California', layout=widgets.Layout(width='120px')),
        widgets.Button(description='🤠 Texas', layout=widgets.Layout(width='120px')),
        widgets.Button(description='🏖️ Florida', layout=widgets.Layout(width='120px')),
        widgets.Button(description='🏙️ Northeast', layout=widgets.Layout(width='120px'))
    ])
    
    # Extraction control buttons  
    extraction_buttons = widgets.HBox([
        widgets.Button(description='🎯 Extract 1 Grid', button_style='primary', layout=widgets.Layout(width='140px')),
        widgets.Button(description='🚀 Extract 3 Grids', button_style='primary', layout=widgets.Layout(width='140px')),
        widgets.Button(description='🌊 Extract Coastal', button_style='primary', layout=widgets.Layout(width='140px')),
        widgets.Button(description='📊 Quick Status', button_style='info', layout=widgets.Layout(width='140px'))
    ])
    
    # Progress display area
    progress_output = widgets.Output(layout=widgets.Layout(height='200px', border='1px solid #ccc'))
    
    # Map display area  
    map_output = widgets.Output(layout=widgets.Layout(height='600px', border='1px solid #ccc'))
    
    # Wire up button functions
    def on_button_click(button):
        with progress_output:
            clear_output()
            print(f"🔄 {button.description} clicked...")
        
        if 'Full Map' in button.description:
            with map_output:
                clear_output()
                show_grid_map(manager.coverage_system)
                
        elif 'Refresh' in button.description:
            with map_output:
                clear_output()
                viz = GridVisualization(manager.coverage_system)
                viz.create_map()
                plt.show()
                
        elif 'Save Map' in button.description:
            timestamp = time.strftime("%Y%m%d_%H%M%S")
            filename = f"grid_progress_{timestamp}.png"
            save_grid_map(manager.coverage_system, filename)
            with progress_output:
                print(f"📸 Map saved as {filename}")
                
        elif 'Heatmap' in button.description:
            with map_output:
                clear_output()
                show_progress_heatmap(manager.coverage_system)
                
        elif 'California' in button.description:
            with map_output:
                clear_output()
                show_california_map(manager.coverage_system)
                
        elif 'Texas' in button.description:
            with map_output:
                clear_output()
                show_texas_map(manager.coverage_system)
                
        elif 'Florida' in button.description:
            with map_output:
                clear_output()
                show_florida_map(manager.coverage_system)
                
        elif 'Northeast' in button.description:
            with map_output:
                clear_output()
                show_northeast_map(manager.coverage_system)
                
        elif 'Extract 1' in button.description:
            with progress_output:
                print("🎯 Starting single grid extraction...")
                # Note: Would need to run await extract_next_grid() in async context
                
        elif 'Extract 3' in button.description:
            with progress_output:
                print("🚀 Starting 3-grid extraction...")
                # Note: Would need to run await extract_multiple_grids_demo(3) in async context
                
        elif 'Coastal' in button.description:
            with progress_output:
                print("🌊 Starting coastal grid extraction...")
                # Note: Would need to run await extract_coastal_grids_demo(2) in async context
                
        elif 'Quick Status' in button.description:
            with progress_output:
                clear_output()
                manager.show_status()
    
    # Connect all buttons
    for button in map_buttons.children + region_buttons.children + extraction_buttons.children:
        button.on_click(on_button_click)
    
    # Create sections
    map_section = widgets.VBox([
        widgets.HTML("<h3>🗺️ Map Controls</h3>"),
        map_buttons
    ])
    
    region_section = widgets.VBox([
        widgets.HTML("<h3>🌍 Regional Focus</h3>"),
        region_buttons
    ])
    
    extraction_section = widgets.VBox([
        widgets.HTML("<h3>⚡ Extraction Controls</h3>"),
        extraction_buttons
    ])
    
    progress_section = widgets.VBox([
        widgets.HTML("<h3>📊 Progress & Status</h3>"),
        progress_output
    ])
    
    map_display_section = widgets.VBox([
        widgets.HTML("<h3>🗺️ Live Map Display</h3>"),
        map_output
    ])
    
    # Organize into tabs or accordion
    control_tabs = widgets.Tab()
    control_tabs.children = [map_section, region_section, extraction_section]
    control_tabs.set_title(0, '🗺️ Maps')
    control_tabs.set_title(1, '🌍 Regions') 
    control_tabs.set_title(2, '⚡ Extract')
    
    # Main interface
    interface = widgets.VBox([
        header,
        control_tabs,
        progress_section,
        map_display_section
    ])
    
    return interface, progress_output, map_output

# Create and display the interface
visual_interface, progress_out, map_out = create_visual_interface()

print("✅ Visual control panel ready!")
print("\n🎮 USAGE:")
print("   1. Run the cell below to display the interactive interface")
print("   2. Use buttons to show maps, focus on regions, and control extraction")
print("   3. Maps show color-coded grids: 🔴 Pending, 🟡 In Progress, 🟢 Completed")
print("   4. For live extraction tracking, use start_live_tracking(manager.coverage_system)")

# Display interface function
def show_visual_interface():
    """Display the complete visual interface"""
    display(visual_interface)
    
    # Show initial status
    with progress_out:
        print("📊 Initial system status:")
        manager.show_status()
    
    # Show initial map
    with map_out:
        print("🗺️ Loading initial grid map...")
        show_grid_map(manager.coverage_system)

print("\n▶️ Call show_visual_interface() to start!")

In [ ]:
# Display areas for map and stats
display(widgets.HTML("<h4>🗺️ Preview Map:</h4>"))
display(map_area)

display(widgets.HTML("<h4>📊 Area Statistics:</h4>"))
display(stats_area)

## 8. Download Implementation with Progress Tracking

In [ ]:
async def download_tiles_with_progress(extractor: OSMTileExtractor, tiles: List[TileInfo], 
                                     batch_size: int = 100) -> DownloadStats:
    """Download tiles with visual progress tracking"""
    
    stats = DownloadStats()
    stats.total_tiles = len(tiles)
    stats.start_time = time.time()
    
    # Create progress bar
    progress_bar = tqdm(total=len(tiles), desc="Downloading tiles")
    
    # Process in batches
    for i in range(0, len(tiles), batch_size):
        batch = tiles[i:i + batch_size]
        batch_start = time.time()
        
        # Download batch
        try:
            results = await download_batch_async(extractor, batch)
            
            # Update statistics
            batch_downloaded = sum(results)
            batch_failed = len(results) - batch_downloaded
            
            stats.downloaded += batch_downloaded
            stats.failed += batch_failed
            
            # Calculate speed
            batch_time = time.time() - batch_start
            batch_speed = len(batch) / batch_time if batch_time > 0 else 0
            stats.speeds.append(batch_speed)
            
            # Update progress bar
            progress_bar.update(len(batch))
            progress_bar.set_postfix({
                'Success': f'{stats.success_rate:.1f}%',
                'Speed': f'{batch_speed:.1f} t/s',
                'Avg': f'{stats.average_speed:.1f} t/s'
            })
            
            # Show dashboard every few batches
            if (i // batch_size + 1) % 5 == 0:
                clear_output(wait=True)
                create_progress_dashboard(stats)
                progress_bar = tqdm(total=len(tiles), initial=i+len(batch), desc="Downloading tiles")
                
        except Exception as e:
            print(f"❌ Batch error: {e}")
            stats.failed += len(batch)
    
    progress_bar.close()
    stats.end_time = time.time()
    
    return stats

async def download_batch_async(extractor: OSMTileExtractor, tiles: List[TileInfo]) -> List[bool]:
    """Download a batch of tiles asynchronously"""
    
    connector = aiohttp.TCPConnector(limit=extractor.max_workers)
    timeout = aiohttp.ClientTimeout(total=30)
    
    async with aiohttp.ClientSession(
        connector=connector,
        timeout=timeout,
        headers={'User-Agent': 'Jupyter-OSM-Intersection-Training-Extractor/1.0'}
    ) as session:
        
        semaphore = asyncio.Semaphore(extractor.max_workers)
        
        async def download_single_tile(tile: TileInfo) -> bool:
            async with semaphore:
                return await download_tile_async(extractor, session, tile)
        
        # Execute all downloads concurrently
        tasks = [download_single_tile(tile) for tile in tiles]
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
        # Handle exceptions
        success_results = []
        for result in results:
            if isinstance(result, Exception):
                success_results.append(False)
            else:
                success_results.append(result)
        
        return success_results

async def download_tile_async(extractor: OSMTileExtractor, session: aiohttp.ClientSession, 
                            tile: TileInfo) -> bool:
    """Download a single tile asynchronously"""
    
    tile_path = extractor.output_dir / "tiles" / tile.filename
    
    # Skip if already exists
    if tile_path.exists():
        return True
    
    server = extractor.get_next_server()
    url = extractor.build_tile_url(server, tile.x, tile.y, tile.zoom)
    
    try:
        # Rate limiting
        await asyncio.sleep(1.0 / server.max_requests_per_second)
        
        async with session.get(url) as response:
            if response.status == 200:
                content = await response.read()
                
                # Save tile
                with open(tile_path, 'wb') as f:
                    f.write(content)
                
                # Save metadata
                tile.downloaded = True
                tile.timestamp = datetime.now().isoformat()
                tile.server_used = server.name
                
                metadata_path = extractor.output_dir / "metadata" / f"tile_{tile.zoom}_{tile.x}_{tile.y}.json"
                with open(metadata_path, 'w') as f:
                    json.dump(asdict(tile), f, indent=2)
                
                return True
            else:
                return False
    
    except Exception:
        return False

print("⚡ Download implementation ready!")

## 9. Main Download Function

In [ ]:
async def run_extraction(bbox: Tuple[float, float, float, float], zoom: int = 16, 
                        max_workers: int = 8, batch_size: int = 100):
    """Main function to run the tile extraction with full progress tracking"""
    
    print(f"🚀 Starting OSM Tile Extraction")
    print(f"📍 Area: {bbox}")
    print(f"🔍 Zoom: {zoom}")
    print(f"👥 Workers: {max_workers}")
    print(f"📦 Batch size: {batch_size}")
    print("=" * 50)
    
    # Initialize extractor
    extractor = OSMTileExtractor(CONFIG['output_dir'], max_workers)
    
    # Generate tile grid
    print("📐 Generating tile grid...")
    tiles = extractor.generate_tile_grid(bbox, zoom)
    print(f"✅ Generated {len(tiles):,} tiles")
    
    # Check existing tiles
    existing_count = 0
    for tile in tiles:
        tile_path = extractor.output_dir / "tiles" / tile.filename
        if tile_path.exists():
            existing_count += 1
    
    remaining_tiles = [t for t in tiles if not (extractor.output_dir / "tiles" / t.filename).exists()]
    
    print(f"📊 Progress: {existing_count:,} / {len(tiles):,} tiles already downloaded")
    print(f"⏳ Remaining: {len(remaining_tiles):,} tiles to download")
    
    if not remaining_tiles:
        print("✅ All tiles already downloaded!")
        return
    
    # Start download
    print("\n🚀 Starting download...")
    stats = await download_tiles_with_progress(extractor, remaining_tiles, batch_size)
    
    # Final report
    print("\n" + "=" * 50)
    print("🎉 EXTRACTION COMPLETE!")
    print("=" * 50)
    print(f"📊 Total tiles: {stats.total_tiles:,}")
    print(f"✅ Downloaded: {stats.downloaded:,}")
    print(f"❌ Failed: {stats.failed:,}")
    print(f"📈 Success rate: {stats.success_rate:.1f}%")
    print(f"⏱️  Total time: {stats.total_time/60:.1f} minutes")
    print(f"⚡ Average speed: {stats.average_speed:.1f} tiles/second")
    
    # Save report
    report_path = save_session_report(stats, extractor.output_dir, bbox)
    
    # Final dashboard
    create_progress_dashboard(stats)
    
    return stats, extractor

# Update the download button handler
async def handle_download():
    """Handle the download process"""
    global current_bbox
    
    if current_bbox is None:
        print("❌ Please preview area and calculate stats first!")
        return
    
    # Run extraction
    stats, extractor = await run_extraction(
        current_bbox,
        zoom_slider.value,
        workers_slider.value,
        CONFIG['batch_size']
    )
    
    return stats, extractor

print("🎯 Main extraction function ready!")

## 10. Quick Start Examples

Here are some ready-to-run examples for different use cases:

In [ ]:
# 🚀 QUICK START VISUAL DEMO
# Launch the complete visual interface

print("🚀 LAUNCHING VISUAL GRID EXTRACTION SYSTEM...")

# Initialize grid if needed
try:
    # Check if grids exist
    progress = manager.coverage_system.get_progress_report()
    if progress['total_grids'] == 0:
        print("📋 No grids found. Initializing grid system...")
        manager.initialize_grid_system()
        print("✅ Grid system initialized!")
    else:
        print(f"✅ Found existing grid system with {progress['total_grids']:,} grids")
except:
    print("📋 Initializing grid system...")
    manager.initialize_grid_system()
    print("✅ Grid system initialized!")

# Show the visual interface
print("🎮 Displaying visual control panel...")
show_visual_interface()

print("\n" + "="*60)
print("🎉 VISUAL GRID SYSTEM IS READY!")
print("="*60)

print("\n🗺️ MAP COLOR LEGEND:")
print("   🔴 RED = Pending (not started)")
print("   🟡 YELLOW = In Progress (currently extracting)")  
print("   🟢 GREEN = Completed (extraction finished)")

print("\n🎮 HOW TO USE:")
print("   1. Click map buttons to view different visualizations")
print("   2. Use region buttons to focus on specific areas")
print("   3. Use extraction buttons to start downloading tiles")
print("   4. Watch the map update in real-time as grids complete")

print("\n⚡ QUICK FUNCTIONS:")
print("   • show_grid_map(manager.coverage_system) - Show full US map")
print("   • show_california_map(manager.coverage_system) - California focus")
print("   • show_progress_heatmap(manager.coverage_system) - Progress heatmap")
print("   • start_live_tracking(manager.coverage_system) - Live tracking during extraction")

print("\n🔄 EXTRACTION FUNCTIONS:")
print("   • await extract_next_grid() - Extract one grid")
print("   • await extract_multiple_grids_demo(3) - Extract 3 grids")
print("   • await extract_coastal_grids_demo(2) - Extract coastal grids")

print("\n📊 MONITORING FUNCTIONS:")
print("   • manager.show_status() - Quick status")
print("   • manager.show_progress() - Full report")
print("   • manager.show_next(5) - Show next 5 grids")

print("\n" + "="*60)
print("🎯 READY TO EXTRACT OSM TILES WITH VISUAL TRACKING!")
print("="*60)

## 11. Run Your Extraction

Choose one of the methods below to start your tile extraction:

In [ ]:
# Grid-Based Tile Extraction Functions
# These functions implement the actual extraction logic using the grid system

async def extract_grid_tiles(grid_square, extractor, output_folder="osm_tiles"):
    """Extract all tiles for a specific grid square"""
    print(f"\n🎯 Extracting Grid: {grid_square.grid_id}")
    print(f"   Location: {grid_square.lat_min:.3f}°N to {grid_square.lat_max:.3f}°N")
    print(f"            {grid_square.lng_min:.3f}°W to {grid_square.lng_max:.3f}°W")
    
    # Calculate tiles in this grid
    tiles_to_extract = []
    
    # Get tile bounds for the grid
    min_tile_x, max_tile_y = extractor.deg2num(grid_square.lat_min, grid_square.lng_min, extractor.zoom_level)
    max_tile_x, min_tile_y = extractor.deg2num(grid_square.lat_max, grid_square.lng_max, extractor.zoom_level)
    
    # Generate all tiles in the grid
    for x in range(min_tile_x, max_tile_x + 1):
        for y in range(min_tile_y, max_tile_y + 1):
            # Determine which state this tile belongs to
            lat, lng = extractor.num2deg(x, y, extractor.zoom_level)
            state, _ = coverage_system.assign_tile_to_state(lat, lng)
            
            tiles_to_extract.append({
                'x': x, 
                'y': y, 
                'z': extractor.zoom_level,
                'state': state,
                'lat': lat,
                'lng': lng
            })
    
    print(f"   Total tiles in grid: {len(tiles_to_extract):,}")
    
    # Extract tiles using the fast extractor
    start_time = time.time()
    success_count = 0
    
    # Process tiles in batches for better progress tracking
    batch_size = 50
    for i in range(0, len(tiles_to_extract), batch_size):
        batch = tiles_to_extract[i:i + batch_size]
        batch_results = await extractor.download_tiles_batch_async(batch)
        success_count += sum(batch_results)
        
        # Update progress
        progress = (i + len(batch)) / len(tiles_to_extract) * 100
        print(f"   Progress: {progress:.1f}% ({success_count:,}/{i + len(batch):,} tiles)")
    
    extraction_time = time.time() - start_time
    success_rate = (success_count / len(tiles_to_extract)) * 100
    
    print(f"✅ Grid {grid_square.grid_id} completed!")
    print(f"   Extracted: {success_count:,}/{len(tiles_to_extract):,} tiles ({success_rate:.1f}%)")
    print(f"   Time: {extraction_time:.1f}s ({success_count/extraction_time:.1f} tiles/sec)")
    
    # Update grid status in database
    coverage_system.mark_grid_completed(grid_square.grid_id, success_count)
    
    return success_count, len(tiles_to_extract)

async def extract_single_grid():
    """Extract tiles for the next available grid"""
    next_grid = coverage_system.get_next_grid()
    if not next_grid:
        print("❌ No pending grids available for extraction")
        return
    
    # Mark grid as in progress
    coverage_system.mark_grid_in_progress(next_grid.grid_id)
    
    try:
        # Extract the grid
        success_count, total_tiles = await extract_grid_tiles(next_grid, extractor)
        print(f"\n📊 Single grid extraction completed: {success_count:,}/{total_tiles:,} tiles")
        
    except Exception as e:
        print(f"❌ Error extracting grid {next_grid.grid_id}: {e}")
        # Reset grid status on error
        coverage_system.mark_grid_in_progress(next_grid.grid_id)  # Will keep as in_progress for retry

async def extract_multiple_grids(num_grids=5):
    """Extract tiles for multiple grids in sequence"""
    print(f"🚀 Starting extraction of {num_grids} grids...")
    
    total_success = 0
    total_tiles = 0
    
    for i in range(num_grids):
        print(f"\n--- Grid {i+1}/{num_grids} ---")
        
        next_grid = coverage_system.get_next_grid()
        if not next_grid:
            print(f"✅ No more grids available. Completed {i} grids.")
            break
        
        coverage_system.mark_grid_in_progress(next_grid.grid_id)
        
        try:
            success_count, grid_total = await extract_grid_tiles(next_grid, extractor)
            total_success += success_count
            total_tiles += grid_total
            
        except Exception as e:
            print(f"❌ Error extracting grid {next_grid.grid_id}: {e}")
            continue
    
    print(f"\n🎉 MULTI-GRID EXTRACTION COMPLETE!")
    print(f"   Total extracted: {total_success:,}/{total_tiles:,} tiles")
    print(f"   Success rate: {(total_success/total_tiles)*100:.1f}%")

async def extract_coastal_grids(num_grids=3):
    """Extract grids that are likely to contain coastal areas (for intersection variety)"""
    print(f"🌊 Starting extraction of {num_grids} coastal grids...")
    
    # Get all pending grids and sort by proximity to coasts
    with sqlite3.connect(coverage_system.db_path) as conn:
        cursor = conn.execute('''
            SELECT grid_id, grid_x, grid_y, lat_min, lat_max, lng_min, lng_max,
                   total_tiles, extracted_tiles, status
            FROM grid_squares
            WHERE status = 'pending'
            ORDER BY grid_y, grid_x  -- Start from south and west (more coastal areas)
            LIMIT ?
        ''', (num_grids * 3,))  # Get more options to filter
        
        results = cursor.fetchall()
    
    # Prefer grids closer to coasts (lower grid_y = southern, extreme grid_x = eastern/western)
    coastal_grids = []
    for result in results:
        grid = GridSquare(
            grid_id=result[0], grid_x=result[1], grid_y=result[2],
            lat_min=result[3], lat_max=result[4], lng_min=result[5], lng_max=result[6],
            total_tiles=result[7], extracted_tiles=result[8], status=result[9]
        )
        
        # Prioritize coastal areas (southern states, east/west coasts)
        is_coastal = (
            grid.lat_min < 35 or  # Southern states
            grid.lng_min > -75 or  # East coast
            grid.lng_max < -115    # West coast
        )
        
        if is_coastal:
            coastal_grids.append(grid)
            if len(coastal_grids) >= num_grids:
                break
    
    # Fall back to any grids if not enough coastal ones
    if len(coastal_grids) < num_grids:
        for result in results[len(coastal_grids):]:
            if len(coastal_grids) >= num_grids:
                break
            grid = GridSquare(
                grid_id=result[0], grid_x=result[1], grid_y=result[2],
                lat_min=result[3], lat_max=result[4], lng_min=result[5], lng_max=result[6],
                total_tiles=result[7], extracted_tiles=result[8], status=result[9]
            )
            coastal_grids.append(grid)
    
    # Extract the selected grids
    total_success = 0
    total_tiles = 0
    
    for i, grid in enumerate(coastal_grids):
        print(f"\n--- Coastal Grid {i+1}/{len(coastal_grids)} ---")
        
        coverage_system.mark_grid_in_progress(grid.grid_id)
        
        try:
            success_count, grid_total = await extract_grid_tiles(grid, extractor)
            total_success += success_count
            total_tiles += grid_total
            
        except Exception as e:
            print(f"❌ Error extracting grid {grid.grid_id}: {e}")
            continue
    
    print(f"\n🌊 COASTAL EXTRACTION COMPLETE!")
    print(f"   Total extracted: {total_success:,}/{total_tiles:,} tiles")
    print(f"   Success rate: {(total_success/total_tiles)*100:.1f}%")

# Grid Progress Monitoring and Reporting
# Real-time monitoring functions for the grid-based extraction system

def show_grid_progress():
    """Display current progress of the grid coverage system"""
    report = coverage_system.generate_report()
    print(report)

def show_next_grids(num_grids=10):
    """Show the next grids scheduled for extraction"""
    print(f"📋 NEXT {num_grids} GRIDS TO EXTRACT:")
    print("=" * 60)
    
    # Get next grids without marking them as in progress
    with sqlite3.connect(coverage_system.db_path) as conn:
        cursor = conn.execute('''
            SELECT grid_id, grid_x, grid_y, lat_min, lat_max, lng_min, lng_max,
                   total_tiles, extracted_tiles, status
            FROM grid_squares
            WHERE status = 'pending'
            ORDER BY grid_y, grid_x
            LIMIT ?
        ''', (num_grids,))
        
        results = cursor.fetchall()
    
    if not results:
        print("❌ No pending grids available")
        return
    
    for i, result in enumerate(results, 1):
        grid_id = result[0]
        lat_min, lat_max = result[3], result[4]
        lng_min, lng_max = result[5], result[6]
        total_tiles = result[7]
        
        # Determine primary state for this grid
        center_lat = (lat_min + lat_max) / 2
        center_lng = (lng_min + lng_max) / 2
        primary_state, distance = coverage_system.assign_tile_to_state(center_lat, center_lng)
        
        print(f"  {i:2}. {grid_id}")
        print(f"      📍 {lat_min:.3f}°N to {lat_max:.3f}°N, {lng_min:.3f}°W to {lng_max:.3f}°W")
        print(f"      🗺️  Primary State: {primary_state} ({total_tiles:,} tiles)")
        print()

def show_state_statistics():
    """Show extraction progress by state"""
    coverage_stats = coverage_system.get_coverage_report()
    
    print("🗺️  EXTRACTION PROGRESS BY STATE")
    print("=" * 50)
    
    # Sort states by total tiles
    sorted_states = sorted(coverage_stats['by_state'], key=lambda x: x['total_tiles'], reverse=True)
    
    print(f"{'State':<15} {'Total Tiles':<12} {'Extracted':<12} {'Progress':<10}")
    print("-" * 50)
    
    for state_info in sorted_states:
        state = state_info['state']
        total = state_info['total_tiles']
        extracted = state_info['extracted_tiles']
        progress = (extracted / total * 100) if total > 0 else 0
        
        print(f"{state:<15} {total:,<12} {extracted:,<12} {progress:>6.1f}%")

def show_recent_activity():
    """Show recent extraction activity"""
    with sqlite3.connect(coverage_system.db_path) as conn:
        # Recent completed grids
        cursor = conn.execute('''
            SELECT grid_id, extracted_tiles, completed_at
            FROM grid_squares
            WHERE status = 'completed' AND completed_at IS NOT NULL
            ORDER BY completed_at DESC
            LIMIT 10
        ''')
        
        recent_grids = cursor.fetchall()
    
    if recent_grids:
        print("⚡ RECENT COMPLETED GRIDS (Last 10)")
        print("=" * 40)
        
        for grid_id, tiles, completed_at in recent_grids:
            print(f"  ✅ {grid_id}: {tiles:,} tiles at {completed_at}")
    else:
        print("⚡ No completed grids yet")

# Quick progress check function
def quick_status():
    """Show a quick status summary"""
    progress = coverage_system.get_progress_report()
    
    total_grids = progress['total_grids']
    completed_grids = progress['completed_grids']
    total_tiles = progress['total_tiles']
    extracted_tiles = progress['extracted_tiles']
    completion_pct = progress['completion_percentage']
    
    print(f"📊 QUICK STATUS")
    print(f"   Grids: {completed_grids:,}/{total_grids:,} ({completed_grids/total_grids*100:.1f}%)")
    print(f"   Tiles: {extracted_tiles:,}/{total_tiles:,} ({completion_pct:.1f}%)")
    
    if progress['recent_tiles_24h'] > 0:
        print(f"   Recent: {progress['recent_tiles_24h']:,} tiles in 24h")
        remaining_tiles = total_tiles - extracted_tiles
        days_remaining = remaining_tiles / progress['daily_rate']
        print(f"   ETA: {days_remaining:.1f} days at current rate")

# Test functions
async def test_single_extraction():
    """Test extracting one grid to verify everything works"""
    print("🧪 Testing single grid extraction...")
    
    # Show what we're about to extract
    next_grid = coverage_system.get_next_grid()
    if next_grid:
        print(f"Will extract: {next_grid.grid_id}")
        print(f"Location: {next_grid.lat_min:.3f}°N to {next_grid.lat_max:.3f}°N")
        print(f"Expected tiles: {next_grid.total_tiles:,}")
        
        user_confirm = input("\nProceed with test extraction? (y/n): ")
        if user_confirm.lower() == 'y':
            await extract_single_grid()
        else:
            print("Test cancelled")
    else:
        print("❌ No grids available for testing")

# Initialize monitoring display
print("✅ Grid monitoring functions loaded!")
print("\nAvailable functions:")
print("  • show_grid_progress() - Full progress report")
print("  • show_next_grids(n) - Show next n grids to extract")
print("  • show_state_statistics() - Progress by state")
print("  • show_recent_activity() - Recent completed grids")
print("  • quick_status() - Quick summary")
print("  • test_single_extraction() - Test extraction workflow")

In [ ]:
def analyze_downloaded_tiles(output_dir: str = "osm_training_data"):
    """Analyze the downloaded tiles and show statistics"""
    
    output_path = Path(output_dir)
    tiles_dir = output_path / "tiles"
    metadata_dir = output_path / "metadata"
    
    if not tiles_dir.exists():
        print(f"❌ No tiles directory found at {tiles_dir}")
        return
    
    # Count files
    tile_files = list(tiles_dir.glob("*.png"))
    metadata_files = list(metadata_dir.glob("*.json"))
    
    # Calculate storage
    total_size_bytes = sum(f.stat().st_size for f in tile_files)
    total_size_mb = total_size_bytes / 1024 / 1024
    total_size_gb = total_size_mb / 1024
    
    # Average file size
    avg_size_kb = (total_size_bytes / len(tile_files) / 1024) if tile_files else 0
    
    print("📊 DOWNLOADED DATASET ANALYSIS")
    print("=" * 40)
    print(f"📁 Output directory: {output_path}")
    print(f"🖼️  Total tiles: {len(tile_files):,}")
    print(f"📄 Metadata files: {len(metadata_files):,}")
    print(f"💾 Total storage: {total_size_mb:.1f} MB ({total_size_gb:.2f} GB)")
    print(f"📏 Average tile size: {avg_size_kb:.1f} KB")
    
    # Load and analyze metadata if available
    if metadata_files:
        print("\n🔍 Analyzing tile metadata...")
        
        # Load sample metadata
        with open(metadata_files[0]) as f:
            sample_metadata = json.load(f)
        
        zoom_level = sample_metadata.get('zoom', 'Unknown')
        print(f"🔍 Zoom level: {zoom_level}")
        
        # Calculate coverage area
        if len(metadata_files) >= 4:  # Need at least 4 corners
            lats, lons = [], []
            for metadata_file in metadata_files[:100]:  # Sample first 100
                try:
                    with open(metadata_file) as f:
                        data = json.load(f)
                        lats.extend([data['lat_min'], data['lat_max']])
                        lons.extend([data['lon_min'], data['lon_max']])
                except:
                    continue
            
            if lats and lons:
                lat_range = max(lats) - min(lats)
                lon_range = max(lons) - min(lons)
                
                print(f"🌍 Coverage area:")
                print(f"   Latitude range: {lat_range:.4f}° ({lat_range * 111:.1f} km)")
                print(f"   Longitude range: {lon_range:.4f}° ({lon_range * 111:.1f} km)")
    
    # Create visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # File size distribution
    if tile_files:
        sizes_kb = [f.stat().st_size / 1024 for f in tile_files[:1000]]  # Sample first 1000
        ax1.hist(sizes_kb, bins=30, alpha=0.7, color='skyblue')
        ax1.set_title('Tile Size Distribution (KB)')
        ax1.set_xlabel('Size (KB)')
        ax1.set_ylabel('Count')
    
    # Storage breakdown pie chart
    if total_size_mb > 0:
        labels = ['Tiles', 'Metadata', 'Other']
        metadata_size = sum(f.stat().st_size for f in metadata_files) / 1024 / 1024
        other_size = max(0, total_size_mb * 0.01)  # Estimate
        
        sizes = [total_size_mb - metadata_size - other_size, metadata_size, other_size]
        colors = ['#3498db', '#2ecc71', '#95a5a6']
        
        ax2.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%')
        ax2.set_title('Storage Breakdown (MB)')
    
    plt.tight_layout()
    plt.show()
    
    # Next steps suggestions
    print("\n🎯 NEXT STEPS FOR ML TRAINING:")
    print("1. 🏷️  Start annotating intersections in the downloaded tiles")
    print("2. 🔄 Use tools like LabelImg, CVAT, or VGG Image Annotator")
    print("3. 🧠 Train your intersection detection model")
    print("4. 📈 Expand dataset based on model performance")
    
    return {
        'total_tiles': len(tile_files),
        'total_size_mb': total_size_mb,
        'avg_size_kb': avg_size_kb,
        'tiles_dir': tiles_dir,
        'metadata_dir': metadata_dir
    }

# Run analysis on current dataset
# Uncomment the line below after downloading tiles:
# analysis_results = analyze_downloaded_tiles()

print("📊 Analysis functions ready! Run analyze_downloaded_tiles() after downloading.")

## 13. Save & Export Functions

In [ ]:
def create_training_manifest(output_dir: str = "osm_training_data"):
    """Create a comprehensive manifest for ML training"""
    
    output_path = Path(output_dir)
    tiles_dir = output_path / "tiles"
    metadata_dir = output_path / "metadata"
    
    if not tiles_dir.exists():
        print(f"❌ No tiles found at {tiles_dir}")
        return
    
    # Collect all tile information
    tiles_info = []
    tile_files = list(tiles_dir.glob("*.png"))
    
    for tile_file in tqdm(tile_files, desc="Processing tiles"):
        # Extract tile info from filename
        parts = tile_file.stem.split('_')  # tile_16_1234_5678
        if len(parts) >= 4:
            zoom, x, y = int(parts[1]), int(parts[2]), int(parts[3])
            
            # Load metadata if available
            metadata_file = metadata_dir / f"tile_{zoom}_{x}_{y}.json"
            metadata = {}
            if metadata_file.exists():
                with open(metadata_file) as f:
                    metadata = json.load(f)
            
            # File info
            file_size = tile_file.stat().st_size
            
            tile_info = {
                'filename': tile_file.name,
                'x': x, 'y': y, 'zoom': zoom,
                'file_size_bytes': file_size,
                'coordinates': {
                    'lat_min': metadata.get('lat_min'),
                    'lat_max': metadata.get('lat_max'),
                    'lon_min': metadata.get('lon_min'),
                    'lon_max': metadata.get('lon_max')
                },
                'downloaded_at': metadata.get('timestamp'),
                'server_used': metadata.get('server_used')
            }
            tiles_info.append(tile_info)
    
    # Create manifest
    manifest = {
        'dataset_info': {
            'name': 'OSM Intersection Detection Training Dataset',
            'created': datetime.now().isoformat(),
            'total_tiles': len(tiles_info),
            'zoom_levels': list(set(t['zoom'] for t in tiles_info)),
            'tile_format': 'PNG',
            'tile_size': '256x256',
            'coordinate_system': 'WGS84',
            'source': 'OpenStreetMap'
        },
        'coverage_stats': {
            'total_size_mb': sum(t['file_size_bytes'] for t in tiles_info) / 1024 / 1024,
            'avg_file_size_kb': np.mean([t['file_size_bytes'] for t in tiles_info]) / 1024,
            'bounding_box': {
                'lat_min': min(t['coordinates']['lat_min'] for t in tiles_info if t['coordinates']['lat_min']),
                'lat_max': max(t['coordinates']['lat_max'] for t in tiles_info if t['coordinates']['lat_max']),
                'lon_min': min(t['coordinates']['lon_min'] for t in tiles_info if t['coordinates']['lon_min']),
                'lon_max': max(t['coordinates']['lon_max'] for t in tiles_info if t['coordinates']['lon_max'])
            } if any(t['coordinates']['lat_min'] for t in tiles_info) else None
        },
        'tiles': tiles_info
    }
    
    # Save manifest
    manifest_path = output_path / "training_manifest.json"
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)
    
    # Also save a CSV version for easy analysis
    df = pd.DataFrame(tiles_info)
    csv_path = output_path / "tiles_catalog.csv"
    df.to_csv(csv_path, index=False)
    
    print(f"📋 Training manifest saved: {manifest_path}")
    print(f"📊 CSV catalog saved: {csv_path}")
    print(f"✅ Dataset ready with {len(tiles_info):,} tiles!")
    
    return manifest_path, csv_path

def export_sample_for_annotation(output_dir: str = "osm_training_data", sample_size: int = 100):
    """Export a random sample of tiles for manual annotation"""
    
    output_path = Path(output_dir)
    tiles_dir = output_path / "tiles"
    sample_dir = output_path / "annotation_sample"
    
    sample_dir.mkdir(exist_ok=True)
    
    tile_files = list(tiles_dir.glob("*.png"))
    
    if len(tile_files) < sample_size:
        sample_size = len(tile_files)
        print(f"⚠️  Only {len(tile_files)} tiles available, sampling all of them.")
    
    # Random sample
    import shutil
    sample_files = random.sample(tile_files, sample_size)
    
    for i, tile_file in enumerate(tqdm(sample_files, desc="Copying sample tiles")):
        new_name = f"sample_{i+1:03d}_{tile_file.name}"
        shutil.copy2(tile_file, sample_dir / new_name)
    
    print(f"📦 Exported {sample_size} tiles to {sample_dir}")
    print(f"🏷️  Ready for annotation with your preferred tool!")
    
    return sample_dir

print("💾 Export functions ready!")

## 🎉 Congratulations!

You now have a complete, interactive OSM tile extraction system! Here's what you can do:

### 🚀 **Getting Started:**
1. Use the **Control Panel** above to select your area
2. **Preview** the area on an interactive map
3. **Calculate statistics** to see download time and storage estimates
4. **Start the download** with visual progress tracking

### 📊 **After Downloading:**
- Run `analyze_downloaded_tiles()` to see dataset statistics
- Run `create_training_manifest()` to prepare for ML training
- Run `export_sample_for_annotation()` to get tiles ready for labeling

### 🎯 **Recommended Workflow:**
1. **Start small** - Try the `test_sf` region first (~400 tiles)
2. **Annotate sample** - Label intersections in 50-100 tiles
3. **Train initial model** - Build your first intersection detector
4. **Scale up** - Download larger areas as needed
5. **Iterate** - Improve model and expand dataset

### 💡 **Pro Tips:**
- **Save this notebook** - It contains all your progress and settings
- **Monitor disk space** - Large extractions can use significant storage
- **Use version control** - Track your annotation and model development
- **Start with urban areas** - More intersections for training

**Happy intersection detecting! 🚗🛣️**